# Notebook 15 – Complete Feature Engineering Workflow

## 1. Complete Feature Engineering Workflow

Feature Engineering converts a clean dataset into useful ML-ready features.

The complete workflow is:

Clean Dataset
↓
Understand Existing Features
↓
Identify Feature Gaps
↓
Create Numerical Features
↓
Create Categorical Features
↓
Create Date/Time Features
↓
Create Aggregation Features
↓
Create Interaction Features
↓
Feature Selection
↓
Feature Importance
↓
Leakage Check
↓
Remove Unnecessary Features
↓
Final ML-Ready Feature Set

The goal is to create useful features while avoiding unnecessary features and data leakage.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

print("Clean Dataset Loaded")
print("Shape:", df.shape)

df.head()

Clean Dataset Loaded
Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Understand Existing Features

Before creating new features, we need to understand the existing columns.

For the Titanic dataset, important features include:

- Pclass – Passenger class
- Sex – Passenger gender
- Age – Passenger age
- SibSp – Number of siblings/spouses
- Parch – Number of parents/children
- Fare – Ticket fare
- Embarked – Port of embarkation
- Survived – Target variable

Understanding the existing features helps us identify useful feature gaps.

In [2]:
print("Columns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


## 3. Identify Potential Feature Gaps

A feature gap exists when the existing columns do not directly represent information that may be useful to the model.

For example, SibSp and Parch separately provide family information, but they do not directly show total family size.

Potential gaps include:

- Total family size
- Whether a passenger travelled alone
- Fare per family member
- Passenger title
- Family-level information
- Interaction between age and passenger class

In [3]:
print("Potential feature gaps identified:")
print("- Family size")
print("- Travelling alone")
print("- Fare per person")
print("- Passenger title")
print("- Ticket-level information")
print("- Age and class interaction")

Potential feature gaps identified:
- Family size
- Travelling alone
- Fare per person
- Passenger title
- Ticket-level information
- Age and class interaction


## 4. Create Numerical Features

Numerical feature engineering creates useful numerical features from existing numerical columns.

Examples:

FamilySize = SibSp + Parch + 1

FarePerPerson = Fare / FamilySize

These features can represent information that is not directly available in the original columns.

In [4]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

print("Numerical features created:")
df[["SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "FarePerPerson"]].head()

Numerical features created:


,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson
0,1,0,7.2500,2,0,3.62500
1,1,0,71.2833,2,0,35.64165
2,0,0,7.9250,1,1,7.92500
3,1,0,53.1000,2,0,26.55000
4,0,0,8.0500,1,1,8.05000


## 5. Create Categorical Features

Categorical feature engineering creates useful categories from existing information.

For example, a passenger's title can be extracted from the Name column.

The title may provide information about social status, gender, or age group.

In [5]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.", expand=False)

print("Categorical feature created: Title")
df[["Name", "Title"]].head()

Categorical feature created: Title


,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr


## 6. Create Date/Time Features

The Titanic dataset does not contain customer or transaction dates.

Therefore, a simple event date is used only to demonstrate date/time feature engineering.

In a real ML project, date features should come from actual business or event dates.

Examples include:
- Year
- Month
- Day
- Day of Week
- Weekend indicator

In [6]:
df["Event_Date"] = pd.to_datetime("1912-04-15")

df["Event_Year"] = df["Event_Date"].dt.year
df["Event_Month"] = df["Event_Date"].dt.month
df["Event_Day"] = df["Event_Date"].dt.day
df["Weekend_Indicator"] = (df["Event_Date"].dt.dayofweek >= 5).astype(int)

print("Date/time features created:")
df[["Event_Date", "Event_Year", "Event_Month", "Event_Day", "Weekend_Indicator"]].head()

Date/time features created:


,Event_Date,Event_Year,Event_Month,Event_Day,Weekend_Indicator
0,1912-04-15,1912,4,15,0
1,1912-04-15,1912,4,15,0
2,1912-04-15,1912,4,15,0
3,1912-04-15,1912,4,15,0
4,1912-04-15,1912,4,15,0


## 7. Create Aggregation Features

Aggregation features summarize information at a group level.

For Titanic, Ticket can be used as a group identifier.

Examples:

- Number of passengers on the same ticket
- Total fare for the ticket
- Average fare for the ticket

These features can represent group-level information.

In [7]:
df["Ticket_Passenger_Count"] = df.groupby("Ticket")["PassengerId"].transform("count")
df["Ticket_Average_Fare"] = df.groupby("Ticket")["Fare"].transform("mean")

print("Aggregation features created:")
df[["Ticket", "Ticket_Passenger_Count", "Ticket_Average_Fare"]].head()

Aggregation features created:


,Ticket,Ticket_Passenger_Count,Ticket_Average_Fare
0,A/5 21171,1,7.2500
1,PC 17599,1,71.2833
2,STON/O2. 3101282,1,7.9250
3,113803,2,53.1000
4,373450,1,8.0500


## 8. Create Interaction Features

Interaction features combine two or more features to represent their relationship.

For example:

Age_Pclass_Interaction = Age × Pclass

This can help a model capture relationships that may not be clear from individual features alone.

In [8]:
df["Age_Pclass_Interaction"] = df["Age"] * df["Pclass"]

df["Sibling_Parent_Difference"] = df["SibSp"] - df["Parch"]

print("Interaction features created:")
df[["Age", "Pclass", "Age_Pclass_Interaction",
    "SibSp", "Parch", "Sibling_Parent_Difference"]].head()

Interaction features created:


,Age,Pclass,Age_Pclass_Interaction,SibSp,Parch,Sibling_Parent_Difference
0,22.0,3,66.0,1,0,1
1,38.0,1,38.0,1,0,1
2,26.0,3,78.0,0,0,0
3,35.0,1,35.0,1,0,1
4,35.0,3,105.0,0,0,0


## 9. Feature Selection

Feature selection means choosing useful features and removing irrelevant or redundant features.

For this workflow, we select features that have meaningful information and exclude identifiers such as PassengerId.

The target variable Survived is also not included as an input feature.

In [9]:
selected_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "FarePerPerson",
    "Title", "Ticket_Passenger_Count",
    "Ticket_Average_Fare", "Age_Pclass_Interaction"
]

X = df[selected_features].copy()
y = df["Survived"]

print("Selected features:", len(selected_features))
print(selected_features)

Selected features: 12
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Title', 'Ticket_Passenger_Count', 'Ticket_Average_Fare', 'Age_Pclass_Interaction']


## 10. Analyze Feature Importance

Feature importance helps us understand which features are useful for a Machine Learning model.

A Random Forest can provide model-based feature importance.

Importance shows predictive usefulness, but it does not prove that a feature causes the target.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_model = pd.get_dummies(X, columns=["Title"], dummy_na=True)
X_model = X_model.fillna(X_model.median(numeric_only=True)).fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importance = pd.DataFrame({
    "Feature": X_model.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance.head(10)

,Feature,Importance
22,Title_Mr,0.171003
10,Age_Pclass_Interaction,0.132749
9,Ticket_Average_Fare,0.109221
1,Age,0.109008
7,FarePerPerson,0.107698
4,Fare,0.100808
19,Title_Miss,0.046581
23,Title_Mrs,0.044129
8,Ticket_Passenger_Count,0.044078
0,Pclass,0.039566


## 11. Check for Feature Leakage

Feature leakage occurs when a feature contains information that would not be available at prediction time.

In this workflow:

- Survived is the target and must not be used as an input feature.
- Future information should not be used.
- Target-based aggregations should be avoided.
- Target encoding must be calculated using training data only.
- Preprocessing should be fitted using training data only.

Features such as FamilySize and Age_Pclass_Interaction do not use the target and are therefore safer features.

In [11]:
leakage_check = [
    feature for feature in selected_features
    if feature.lower() in ["survived"]
]

print("Potential target leakage features:", leakage_check)

if not leakage_check:
    print("No direct target feature included.")

Potential target leakage features: []
No direct target feature included.


## 12. Remove Unnecessary Features

Unnecessary features may include identifiers, duplicate information, or features that do not provide useful predictive value.

PassengerId is an identifier rather than a meaningful predictive feature, so it is removed.

Name can also be removed after extracting the useful Title feature.

In [12]:
remove_features = ["PassengerId", "Name", "Ticket", "Cabin", "Event_Date"]

df_final = df.drop(columns=remove_features, errors="ignore")

print("Removed unnecessary columns:")
print(remove_features)

print("\nFinal shape:", df_final.shape)

Removed unnecessary columns:
['PassengerId', 'Name', 'Ticket', 'Cabin', 'Event_Date']

Final shape: (891, 20)


## 13. Create Final Feature Dataset

The final feature dataset should contain the selected and engineered features that are suitable for Machine Learning.

Categorical features need to be encoded, and numerical missing values need to be handled.

The target variable is kept separately from the input features.

In [13]:
final_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "FarePerPerson",
    "Title", "Ticket_Passenger_Count",
    "Ticket_Average_Fare", "Age_Pclass_Interaction"
]

final_X = df[final_features].copy()
final_X = pd.get_dummies(final_X, columns=["Title"], dummy_na=True)
final_X = final_X.fillna(final_X.median(numeric_only=True)).fillna(0)

final_dataset = pd.concat(
    [final_X, df["Survived"]],
    axis=1
)

print("Final ML-ready feature dataset:")
print("Shape:", final_dataset.shape)

final_dataset.head()

Final ML-ready feature dataset:
Shape: (891, 30)


,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,Ticket_Passenger_Count,Ticket_Average_Fare,...,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess,Title_nan,Survived
0,3,22.0,1,0,7.2500,2,0,3.62500,1,7.2500,...,False,False,True,False,False,False,False,False,False,0
1,1,38.0,1,0,71.2833,2,0,35.64165,1,71.2833,...,False,False,False,True,False,False,False,False,False,1
2,3,26.0,0,0,7.9250,1,1,7.92500,1,7.9250,...,False,False,False,False,False,False,False,False,False,1
3,1,35.0,1,0,53.1000,2,0,26.55000,2,53.1000,...,False,False,False,True,False,False,False,False,False,1
4,3,35.0,0,0,8.0500,1,1,8.05000,1,8.0500,...,False,False,True,False,False,False,False,False,False,0


## 14. Document All Feature Engineering Decisions

Every important engineered feature should be documented using:

### Feature Name
Name of the newly created feature.

### Source Columns
Which columns were used?

### Logic
Formula or business rule used to create the feature.

### Reason
Why was the feature created?

### Business Meaning
What does the feature represent?

### ML Relevance
How could it help a Machine Learning model?

### Leakage Check
Could the feature introduce data leakage?

### Final Decision
- Retain
- Remove
- Needs Further Analysis

In [14]:
documentation = pd.DataFrame([
    ["FamilySize", "SibSp, Parch", "SibSp + Parch + 1",
     "Represent family size", "Total family members",
     "May capture family-related patterns",
     "No target used", "Retain"],

    ["IsAlone", "FamilySize", "FamilySize == 1",
     "Identify solo passengers", "Passenger travelling alone",
     "May capture survival patterns",
     "No target used", "Retain"],

    ["FarePerPerson", "Fare, FamilySize", "Fare / FamilySize",
     "Represent fare per person", "Approximate individual fare",
     "May provide useful fare information",
     "No target used", "Needs Further Analysis"],

    ["Title", "Name", "Extract text between comma and period",
     "Capture passenger title", "Social/title information",
     "May represent demographic information",
     "No target used", "Retain"],

    ["Ticket_Passenger_Count", "Ticket, PassengerId",
     "Count passengers per ticket",
     "Capture ticket group size", "Number of passengers sharing ticket",
     "May capture group-related patterns",
     "Calculate without target information", "Retain"],

    ["Age_Pclass_Interaction", "Age, Pclass", "Age * Pclass",
     "Capture feature interaction", "Combined age and class information",
     "May capture non-linear relationships",
     "No target used", "Retain"]
], columns=[
    "Feature Name", "Source Columns", "Logic", "Reason",
    "Business Meaning", "ML Relevance",
    "Leakage Check", "Final Decision"
])

documentation

,Feature Name,Source Columns,Logic,Reason,Business Meaning,ML Relevance,Leakage Check,Final Decision
0,FamilySize,"SibSp, Parch",SibSp + Parch + 1,Represent family size,Total family members,May capture family-related patterns,No target used,Retain
1,IsAlone,FamilySize,FamilySize == 1,Identify solo passengers,Passenger travelling alone,May capture survival patterns,No target used,Retain
2,FarePerPerson,"Fare, FamilySize",Fare / FamilySize,Represent fare per person,Approximate individual fare,May provide useful fare information,No target used,Needs Further Analysis
3,Title,Name,Extract text between comma and period,Capture passenger title,Social/title information,May represent demographic information,No target used,Retain
4,Ticket_Passenger_Count,"Ticket, PassengerId",Count passengers per ticket,Capture ticket group size,Number of passengers sharing ticket,May capture group-related patterns,Calculate without target information,Retain
5,Age_Pclass_Interaction,"Age, Pclass",Age * Pclass,Capture feature interaction,Combined age and class information,May capture non-linear relationships,No target used,Retain


## 15. Clean Dataset → Engineered Dataset → Selected Features → Final ML-Ready Feature Set

The complete transformation is:

**Clean Dataset**
→ Original Titanic features

**Engineered Dataset**
→ Numerical, categorical, date/time, aggregation, and interaction features

**Selected Features**
→ Relevant features retained after feature selection and leakage checks

**Final ML-Ready Feature Set**
→ Encoded and prepared features ready for Machine Learning

This workflow provides a structured way to create useful features while reducing unnecessary features and preventing leakage.

In [15]:
print("Clean Dataset:", df.shape)
print("Engineered Dataset:", df.shape)
print("Selected Features:", len(selected_features))
print("Final ML-Ready Dataset:", final_X.shape)

print("\nWorkflow completed successfully.")

Clean Dataset: (891, 25)
Engineered Dataset: (891, 25)
Selected Features: 12
Final ML-Ready Dataset: (891, 29)

Workflow completed successfully.


## 16. Conclusion

This notebook demonstrated a complete Feature Engineering workflow using the Titanic dataset.

We:

- Loaded the clean dataset.
- Understood existing features.
- Identified feature gaps.
- Created numerical features.
- Created categorical features.
- Demonstrated date/time features.
- Created aggregation features.
- Created interaction features.
- Performed feature selection.
- Analyzed feature importance.
- Checked for feature leakage.
- Removed unnecessary features.
- Created the final ML-ready feature dataset.
- Documented important feature engineering decisions.

The main goal is to create features that are meaningful, useful for Machine Learning, and safe from data leakage.

In [16]:
print("Complete Feature Engineering Workflow finished successfully.")
print("Final dataset is ready for Machine Learning.")

Complete Feature Engineering Workflow finished successfully.
Final dataset is ready for Machine Learning.
